# 🤖 RefillCare — Phase 4: Machine Learning Model Training & Evaluation

## 1. Objective
Phase 4 trains and evaluates gradient boosted decision trees (XGBoost, HistGradientBoosting) and Random Forest regressors against the baseline to predict `days_until_next_purchase`, evaluates subgroup accuracy, and converts predictions into actionable `expected_refill_date` values.

**Dataset context:** retrained on the expanded ~5.8-year history (`2020-12-24` → `2026-08-31`) using temporal splits Train ≤ `2026-04-30` (~465.6k rows), Val `2026-05`–`2026-06` (~12.8k), Test `2026-07`–`2026-08` (~7.5k).


## 2. Input Data
We load the trained model artifact (`refill_model.joblib`) and Phase 4 model report (`phase4_model_report.json`).


In [1]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np

# Universal workspace root and sys.path resolver
# Finds the root directory containing 'refillcare/__init__.py' regardless of where kernel is started
current_dir = Path(__file__).resolve().parent if "__file__" in locals() else Path.cwd()
project_root = current_dir.resolve()
while project_root.parent != project_root and not (project_root / "refillcare" / "__init__.py").exists():
    project_root = project_root.parent

if (project_root / "refillcare" / "__init__.py").exists() and str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Safe display helper for Jupyter and standalone environments
try:
    from IPython.display import display
except ImportError:
    display = print

# Safe matplotlib import
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

def find_file(relative_path: str) -> Path:
    candidates = [
        project_root / relative_path,
        Path.cwd() / relative_path,
        Path("..") / relative_path,
        Path("../..") / relative_path,
        Path(r"C:\Users\sunil\ai-mediastra-whatsapp-reminder\ai-mediastra-whatsapp-reminder") / relative_path,
    ]
    for c in candidates:
        if c.exists():
            return c.resolve()
    return candidates[0]

import json
import joblib

model_path = find_file("data/refillcare/processed/models/refill_model.joblib")
report_path = find_file("data/refillcare/processed/phase4_model_report.json")

bundle = joblib.load(model_path)
with open(report_path, "r") as f:
    report = json.load(f)

print(f"Loaded Selected Model: {report['selected_model']}")


Loaded Selected Model: XGBoost


C:\Users\sunil\AppData\Local\Programs\Python\Python313\Lib\pickle.py:1760: UserWarning: [17:11:52] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\gbm\../common/error_msg.h:83: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  setstate(state)


## 3. Processing & Model Comparison
We compare candidate models on the validation set.


In [2]:
# Model comparison summary table
comp_data = []
for m_name, m_metrics in report["validation_comparison"].items():
    comp_data.append({
        "Model": m_name,
        "Val MAE (days)": m_metrics["mae"],
        "Val RMSE (days)": m_metrics["rmse"],
        "Val R2": m_metrics["r2"],
        "Within +-3d (%)": m_metrics["within_3_days_pct"],
        "Within +-7d (%)": m_metrics["within_7_days_pct"],
    })

comp_df = pd.DataFrame(comp_data).sort_values("Val MAE (days)")
display(comp_df)


,Model,Val MAE (days),Val RMSE (days),Val R2,Within +-3d (%),Within +-7d (%)
0,HistoricalMedianBaseline,31.25,115.91,-32.5041,29.15,46.50
2,XGBoost,42.77,68.35,-10.6512,9.86,23.10
3,HistGradientBoosting,42.83,68.44,-10.6794,9.70,22.36
1,RandomForest,42.92,69.03,-10.8835,9.58,23.33


## 4. Results & Deep-Dive Analysis
We inspect feature importances and subgroup performance by refill interval and history depth.


In [3]:
# 4.1 Top Feature Importances Plot
top_feats = pd.DataFrame(report["top_features"]).head(10)

if plt is not None:
    plt.figure(figsize=(10, 5))
    plt.barh(top_feats["feature"][::-1], top_feats["importance"][::-1], color="#3867d6", edgecolor="black")
    plt.title(f"Top 10 Feature Importances ({report['selected_model']})", fontsize=13, pad=12)
    plt.xlabel("Importance Weight", fontsize=11)
    plt.grid(axis="x", linestyle="--", alpha=0.7)
    plt.tight_layout()
    plt.show()
else:
    print(top_feats)


                        feature  importance
0  has_multiple_prior_purchases    0.339621
1         purchase_count_so_far    0.120007
2             is_first_purchase    0.047999
3       historical_interval_max    0.045456
4          salt_category_PHARMA    0.043538
5           salt_itemcat_PHARMA    0.038433
6    historical_interval_median    0.033743
7      historical_interval_mean    0.026931
8        salt_category_JUNCTION    0.023806
9           salt_itemcat_FRIDGE    0.021300


In [4]:
# 4.2 Subgroup Performance by History Depth
hist_data = []
for k, v in report["test_subgroups_by_history_length"].items():
    hist_data.append({
        "History Depth": k,
        "Test Events Count": v["count"],
        "Test MAE (days)": v["mae"],
        "Within +-7d (%)": v["within_7_days_pct"],
    })
display(pd.DataFrame(hist_data))


,History Depth,Test Events Count,Test MAE (days),Within +-7d (%)
0,2_purchases,1282,133.15,0.00
1,3_to_5_purchases,1051,57.44,1.90
2,gt_5_purchases,5155,17.91,33.79


In [5]:
# 4.3 Interactive Inference Demo: Predicting Expected Refill Date
from refillcare.models.prediction import predict_refill_date

sample_patient = {
    "customerId": "RAMESH_9849012345",
    "itemId": "101",
    "itemName": "TELMISARTAN-40MG",
    "invoice_date": pd.Timestamp("2026-06-15"),
    "purchase_count_so_far": 6,
    "days_since_first_purchase": 150,
    "days_since_previous_purchase": 30.0,
    "historical_interval_median": 30.0,
    "historical_interval_mean": 29.8,
    "historical_interval_std": 2.1,
    "historical_interval_min": 28.0,
    "historical_interval_max": 32.0,
    "historical_interval_cv": 0.07,
    "quantity": 30,
    "freeQuantity": 0,
    "avg_historical_quantity": 30.0,
    "quantity_vs_avg_ratio": 1.0,
    "purchase_month": 6,
    "purchase_day_of_week": 0,
    "purchase_day_of_month": 15,
    "purchase_day_of_year": 166,
    "purchase_quarter": 2,
    "is_weekend": 0,
    "is_first_purchase": 0,
    "has_multiple_prior_purchases": 1,
    "is_recurring_history": 1,
    "therapeuticCategory": "CARDIAC",
    "salt_category": "TABLETS",
    "salt_itemcat": "PHARMA",
}

pred_res = predict_refill_date(bundle, sample_patient)
print("=== RefillCare Prediction Output ===")
for k, v in pred_res.items():
    print(f"  {k:30}: {v}")


=== RefillCare Prediction Output ===
  customerId                    : RAMESH_9849012345
  itemId                        : 101
  itemName                      : TELMISARTAN-40MG
  current_purchase_date         : 2026-06-15
  predicted_days_until_refill   : 52.0
  expected_refill_date          : 2026-08-06
  refill_confidence             : HIGH
  purchase_count_so_far         : 6


## 5. What This Means
- **Retrained on full ~5.8-year data:** Artifacts now use Train `2020-12-24`→`2026-04-30`, Val `2026-05`→`2026-06`, Test `2026-07`→`2026-08` (not the old 1-year extract).
- **Selected model:** XGBoost (Validation MAE **42.77** days; Test MAE **43.19** days; $\pm 7$d accuracy **23.53%** on test).
- **Baseline note:** On this longer history, Historical Median Baseline Validation MAE is **31.25** days — currently stronger than tree models overall; treat XGBoost as the serialized ML candidate while we tune for long-tail intervals.
- **Better on deep histories:** For patients with $>5$ historical purchases, Test MAE improves to **17.91** days ($\pm 7$d: **33.79%**).
- **Actionable Reminder Dates:** Converting predicted days to `expected_refill_date` provides the target date for WhatsApp reminder triggers (e.g. -7d, -3d, -1d).


## 6. Conclusion
Phase 4 was re-run on the expanded dataset and saved a new serialized model (`refill_model.joblib`) plus `phase4_model_report.json`. Restart the notebook kernel and re-run cells to load the updated artifacts.
